In [1]:
import sys
import os

sys.path.append('/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit')
sys.path.append('/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/neutron')
sys.path.append('/home/paule/open_mc_projects/MC-1D_DT')



from src.geometry_classes import Geometry, Material, Neutron, Source, _BatchSource
import src.geometry_classes as geom
import src.performance_classes as perf
import src.tally_classes as tally

import src.export_simulation_v3 as xpcsv
import src.export_print_csv as xpcsv

import openmc

openmc.config['cross_sections'] = "/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/cross_sections.xml"


data_folder = '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/wmp'
file_h5_test = '092235.h5'
multipole_data = openmc.data.WindowedMultipole.from_hdf5(os.path.join(data_folder, file_h5_test))


nuclide_name = [ "U238", "U235", "Pu239","Pu240", "Cnat", "O16", "H1", "Fe56", "Xe135", "Na23" ]
nuclide_list = [ '092238', '092235', '094239', '094240', '006000',  '008016', '001001','026056','054135', '011023']
nuclides = {}

for nuclide_id, name in zip(nuclide_list, nuclide_name):
    file_h5 = nuclide_id + '.h5'
    nuclides[name] = openmc.data.WindowedMultipole.from_hdf5(
        os.path.join(data_folder, file_h5)
    )


['/home/paule/anaconda3/envs/vectfit39/lib/python39.zip', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9/lib-dynload', '', '/home/paule/anaconda3/envs/vectfit39/lib/python3.9/site-packages', '/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit', '/home/paule/open_mc_projects/xs_lib/endfb-vii.1-hdf5/neutron', '/home/paule/open_mc_projects/MC-1D_DT', '/home/paule/open_mc_projects/windowed_multipole/02_working_notebook_vectfit']


In [2]:
# One slab of U238 — simple test case
U238 = nuclides['U238']
U235 = nuclides['U235']

# ============================================================================
# Uranium atom density
# ============================================================================
# Uranium metal density ~ 19.1 g/cm³,
# N = rho * Na / A  (atoms/cm³)
rho_U   = 19.1               # g/cm³
NA      = 6.02214076e23      # atoms/mol

x_U235 = 0.00                # 0% enrichment
x_U238 = 1.0 - x_U235

M_U8   = 238.05078826        # g/mol
M_U5   = 235.0439299         # g/mol
N_total_U8 = rho_U * NA / (x_U238 * M_U8 + x_U235 * M_U5)
N_total_U5 = rho_U * NA / (x_U235 * M_U5 + x_U238 * M_U8)

N_U235 = 1.0 * N_total_U5   # 0%  enrichment
N_U238 = 1.0 * N_total_U8   # 100%

# ============================================================================
# Materials
# ============================================================================
slab1 = Material(
    name     = "cell 1",
    nuclides = [(U238, N_U238)],
    T        = 293.6,    # K (~20 °C)
)

slab2 = Material(
    name     = "cell 2",
    nuclides = [(U238, N_U238)],
    T        = 2000,     # K
)

# ============================================================================
# Geometry  (two slabs)
# ============================================================================
geom = Geometry(majorant_log=True, verbose=False)

geom.add_material(slab1)
geom.add_material(slab2)
geom.boundaries     = [0.0, 2.0, 15.0]  # cm
geom.material_array = [slab1, slab2]

# Flux tally: 5 energy groups
energy_bins = [10.0, 50.0, 600.0, 1e4, 1e6, 2e7]  # eV
geom.attach_flux_tally(energy_bins)

# Verification tally: same energy bins, surface at slab1/slab2 interface
geom.attach_verification_tally(
    energy_bins = energy_bins,
    surface_xs  = [2.0],        # slab1/slab2 interface
)

# Majorant and XS method
geom.set_mode("validation", filename='validation/xs_generation/statepoint.200.h5')
#geom.df_group_xs = df_cross_section

# ============================================================================
# Source  (monoenergetic, at x=0, pointing right)
# ============================================================================
N_NEUTRONS  = 100   # neutrons per batch
N_BATCHES   = 5    # number of independent batches (100 neutrons each)

src = Source(
    neutron_nbr  = N_NEUTRONS,
    energy_range = [500, 150],
    energy_dist  = "mono",
    position     = [0.0, 0.0, 0.0],
    direction    = [1.0, 0.0, 0.0],
)

# ============================================================================
# Run — batch mode
# ============================================================================
batch_stats = geom.run_batch(src, N_BATCHES)
xpcsv.export_cross_batch_stats(batch_stats, geom, print_to_console=True)
# ============================================================================
# Results
# ============================================================================
#print(geom.summary())



U238
U238
15.0
  Batch   1/5  (100 neutrons)  wall=1.464s
  Batch   2/5  (100 neutrons)  wall=1.575s
  Batch   3/5  (100 neutrons)  wall=1.531s
  Batch   4/5  (100 neutrons)  wall=1.157s
  Batch   5/5  (100 neutrons)  wall=1.595s

  CROSS-BATCH STATISTICS

  FLUX TALLY [cm · src-n⁻¹]
  Group                               Mean           ±Std    Rel.Err
  -----------------------------------------------------------------
  10-50 eV                      0.0000e+00     0.0000e+00        inf
  50-600 eV                     1.2974e+01     7.0223e-01     0.0541
  600-10000 eV                  0.0000e+00     0.0000e+00        inf
  10000-1000000 eV              0.0000e+00     0.0000e+00        inf
  1000000-20000000 eV           0.0000e+00     0.0000e+00        inf

  ABSORPTION RATE [reactions · src-n⁻¹]
  Region / Group                      Mean           ±Std    Rel.Err
  -----------------------------------------------------------------
  0.0-2.0 cm | 10-50 eV         0.0000e+00     0.0000e+